## Example for extracting data for GPT prompting

### This is not the final/complete code but more about how to get the desired data from the table

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import binom
import copy

## andmetabelid

In [2]:
filename = "../drive_data/v33_koondkorpus_transaktsioonid.db"
conn = sqlite3.connect(filename)
cursor = conn.cursor()

## graafiku punktide info

In [3]:
query = """SELECT verb, verb_compound, morph_case, ratio, level, not_ann_words, ann_words,unique_lemmas, ann_unique_lemmas, not_ann_unique_lemmas, olulisus
            FROM lines_class_info3
            """

class_info = pd.read_sql(query, conn)
class_info

,verb,verb_compound,morph_case,ratio,level,not_ann_words,ann_words,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus
0,aasima,,ad,-9.965784,-,7.0,NaN,1,NaN,5.0,-
1,abistama,,abl,-9.965784,-,6.0,NaN,1,NaN,4.0,-
2,abistama,,all,-9.965784,-,15.0,NaN,1,NaN,12.0,-
3,adresseerima,,in,-9.965784,-,6.0,NaN,1,NaN,3.0,-
4,aeglustama,,all,-9.965784,-,8.0,NaN,1,NaN,7.0,-
...,...,...,...,...,...,...,...,...,...,...,...
21264,õnnestuma,,in,1.227736,-,1286.0,541.0,312,209.0,529.0,-
21265,õppima,,in,4.905379,n90,5848.0,5724.0,547,465.0,968.0,0.0
21266,ütlema,,ad,-5.088166,n10,9790.0,362.0,459,112.0,1118.0,0.0
21267,ütlema,,el,0.974385,n70,2839.0,949.0,629,337.0,1192.0,0.97997


## näitelausete tabel

In [17]:
#query = f"SELECT * FROM spatial_obl"

#spatial_obl = pd.read_sql(query, conn)
#spatial_obl

## võtta ainult n80 tsooni punktid

In [4]:
filtered_class = class_info[class_info["level"]=="n80"]
filtered_class = filtered_class.sort_values(["olulisus"])

In [5]:
filtered_class

,verb,verb_compound,morph_case,ratio,level,not_ann_words,ann_words,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus
21249,valitsema,,in,3.044781,n80,3776.0,1865.0,677,551.0,1234.0,0.0
21223,toimuma,,in,2.873490,n80,24937.0,22095.0,2200,1871.0,4539.0,0.0
19354,treenima,,in,3.900242,n80,387.0,433.0,142,124.0,198.0,0.0
19486,õpetama,,in,3.626783,n80,788.0,630.0,219,181.0,317.0,0.0
19531,kasvama,üles,in,3.798366,n80,391.0,320.0,172,158.0,136.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
18864,põgenema,,adit,4.285402,n80,231.0,351.0,83,72.0,90.0,8.0e-05
19931,elutsema,,in,4.554589,n80,150.0,94.0,74,70.0,105.0,8.0e-05
19940,müüma,,ill,4.610498,n80,84.0,171.0,74,67.0,53.0,8.0e-05
19943,pöörduma,,adit,4.643856,n80,416.0,325.0,74,63.0,152.0,8.0e-05


## json formaat

In [6]:
messages = [{
            "role": "system",
            "content": (
                "Sa oled assistent, kes töötleb JSON-massiivi objekte, "
                "kus iga objekt sisaldab võtmeid 'lause' (täislause) ja 'clause' (märgitud fraas). "
                "Sinu ülesanne on otsustada, kas iga 'clause' viitab *asukohale* "
                "(näiteks linn, riik, piirkond, maamärk, park, aadress jne). \n\n"
                "Väljund peab olema JSON-massiiv, mis sisaldab ainult sõnu 'yes' või 'no', "
                "üks vastus iga sisendobjekti kohta, samas järjekorras nagu sisendis. "
                "Ei tohi lisada selgitusi, kirjavahemärke, tühikuid ega muud teksti. "
                "Näide:\n"
                "Sisend: [{\"lause\": \"Me läksime Pariisi\", \"clause\": \"Pariisi\"}, "
                "{\"lause\": \"Ta alustas tööd kell üheksa\", \"clause\": \"kell üheksa\"}]\n"
                "Väljund: [\"yes\", \"no\"]"
            ),
            }]
input_json = {
                "lause" : "",
                "clause" : ""
            }

## võtta spatial_obl tabelist näitelaused koos vajaliku infoga

kui tahta ainult location näiteid, siis peaks from sees olema lisatingimus

verb = '{v}' and 
verb_compound='{v_c}' and 
morph_case='{m_c}'
and ekilex_tag = 'location'

In [7]:
# nt class_info tabelist esimene ja sellele vastavad näited

#for i in range(len(filtered_class)):
#example_row = filtered_class.iloc[i]

example_row = filtered_class.iloc[4]

v = example_row["verb"]
v_c = example_row["verb_compound"]
m_c = example_row["morph_case"]

# transaction_head.form as head_form, lemma, spatial_obl.form as verb_form, verb, verb_compound, morph_case, sentence_id, sentence, phrase
query = f"""SELECT head_id, head_form, head_lemma, tbl2.form as verb_form, tbl1.verb, tbl1.verb_compound, 
            tbl1.morph_case, tbl1.sentence_id, tbl1.sentence, tbl2.phrase, tbl1.ekilex_tag

            FROM (
            SELECT head_id, form as head_form, lemma as head_lemma, verb, verb_compound, morph_case, 
            sentence_id, sentence, ekilex_tag 
            FROM spatial_obl 
            where 
            verb = '{v}' and 
            verb_compound='{v_c}' and 
            morph_case='{m_c}'
            ) as tbl1

            join
            
            (SELECT * from transaction_head) as tbl2 on 
            tbl1.verb = tbl2.verb and 
            tbl1.verb_compound = tbl2.verb_compound and 
            tbl1.sentence_id = tbl2.sentence_id
            """

spatial_obl_ex = pd.read_sql(query, conn)
spatial_obl_ex
# kui tahta kõiki näiteid anda gpt-le
for j in range(len(spatial_obl_ex)):
    ex = spatial_obl_ex.iloc[j]
    peasona = ex["head_form"]
    phrase = ex["phrase"]
    input_json["lause"] = ex["sentence"]
    input_json["clause"] = peasona
    break

    #break
    
    
#print(input_json)    
messages.append({"role":"user", "content":str(input_json)})   
    
spatial_obl_ex

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag
0,6368,teadmises,teadmine,kasvas,kasvama,üles,in,3760,"Ema laulja , isa Endel Padrik , näitleja ja ka...",laulja nii kasvas Jaana üles teadmises,state
1,26003,Portlandis,Portlandi,Kasvasin,kasvama,üles,in,15017,"Kasvasin üles Portlandis , Maine'is ja mul oli...",Kasvasin üles Portlandis,None
2,46970,Bronxis,Bronxi,kasvasin,kasvama,üles,in,27255,"Ma kasvasin üles Bronxis , ma elasin seal 20 a...",Ma kasvasin üles Bronxis,None
3,54396,Aafrikas,Aafrika,kasvanud,kasvama,üles,in,31692,Aga Aafrikas üles kasvanud poisina ei teadnud ...,Aafrikas üles kasvanud,location
4,90437,vaimus,vaim,kasvasid,kasvama,üles,in,52307,Toomas ja temast poolteist aastat noorem vend ...,Toomas kasvasid üles vaimus,None
...,...,...,...,...,...,...,...,...,...,...,...
729,28591021,koolides,kool,kasvanud,kasvama,üles,in,18936306,"KB : On veel inimesi , kes kasvanud üles KGB ...",kes kasvanud üles koolides,None
730,28611307,külades,küla,kasvanud,kasvama,üles,in,18948958,Suur osa Eesti tippsportlasi elab küll Tartus ...,on üles kasvanud külades,None
731,28701500,Eestis,Eesti,kasvanud,kasvama,üles,in,19018175,"Võin kinnitada , et vaatamata sellele , et väg...",ajal Eestis üles kasvanud,location
732,28742481,Eestis,Eesti,kasvanud,kasvama,üles,in,19046659,Aga kuidasmoodi te selgitate sellist olukorda ...,kes ei ole üles kasvanud Eestis,location


In [8]:
for m in messages:
    print(m, "\n")

{'role': 'system', 'content': 'Sa oled assistent, kes töötleb JSON-massiivi objekte, kus iga objekt sisaldab võtmeid \'lause\' (täislause) ja \'clause\' (märgitud fraas). Sinu ülesanne on otsustada, kas iga \'clause\' viitab *asukohale* (näiteks linn, riik, piirkond, maamärk, park, aadress jne). \n\nVäljund peab olema JSON-massiiv, mis sisaldab ainult sõnu \'yes\' või \'no\', üks vastus iga sisendobjekti kohta, samas järjekorras nagu sisendis. Ei tohi lisada selgitusi, kirjavahemärke, tühikuid ega muud teksti. Näide:\nSisend: [{"lause": "Me läksime Pariisi", "clause": "Pariisi"}, {"lause": "Ta alustas tööd kell üheksa", "clause": "kell üheksa"}]\nVäljund: ["yes", "no"]'} 

{'role': 'user', 'content': "{'lause': 'Ema laulja , isa Endel Padrik , näitleja ja kauaaegne raadiodiktor - nii kasvas väike Jaana üles teadmises , et kuidagipidi peab tema elu ikka loominguga seotud olema .', 'clause': 'teadmises'}"} 



In [9]:
conn.close()